# Portfolio Construction

This exercise will focus on finding the best portfolio as well as run test to see which one is the best and suitable for our portfolio

#### We'll test 2 candidate strategies:
##### A) Defensive ESG + LowVol + (light) Momentum (core-satellite scoring)
##### B) ESG Best-in-class + Minimum Variance optimization

### Import the necessary library

In [1]:
pip install refinitiv-data pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import datetime as dt
import numpy as np
import pandas as pd
import refinitiv.data as rd

from config import CFG
from backtest import run_backtest


In [3]:
# df_esg_western_eu = rd.get_data(...)  # như bạn làm trước đó

meta = df_esg_western_eu.copy()

if "Instrument" in meta.columns:
    meta = meta.rename(columns={"Instrument": "ric"})
elif "ric" not in meta.columns:
    meta = meta.reset_index().rename(columns={"index": "ric"})

if "ESG Score" in meta.columns:
    meta = meta.rename(columns={"ESG Score": "esg"})
else:
    meta = meta.rename(columns={"TR.TRESGScore": "esg"})

meta["esg"] = pd.to_numeric(meta["esg"], errors="coerce")
meta = meta.dropna(subset=["ric","esg"]).drop_duplicates(subset=["ric"])
meta = meta.set_index("ric")[["esg"]].sort_values("esg", ascending=False)

meta.shape

NameError: name 'df_esg_western_eu' is not defined

In [ ]:
import datetime as dt
import pandas as pd
import refinitiv.data as rd

# Assuming rics, meta, CFG are defined earlier in your script
rics = meta.index.tolist()

end_date = dt.date.today()
start_date = end_date - dt.timedelta(days=CFG["lookback_days_prices"] * 2)

rd.open_session()

# --- FIX 1: Add the date field explicitly ---
raw_px = rd.get_data(
    universe=rics,
    fields=["TR.PriceClose.date", "TR.PriceClose"], 
    parameters={
        "SDate": start_date.isoformat(),
        "EDate": end_date.isoformat(),
        "Frq": "D"
    }
)

rd.close_session()

# Dynamically find the date column (usually comes back as 'Date')
date_col = next((c for c in raw_px.columns if "date" in str(c).lower()), None)
if date_col is None:
    raise KeyError(f"Không thấy cột date trong raw_px. Columns hiện có: {list(raw_px.columns)}")

# --- FIX 2: Check the exact column name for Price Close ---
# Refinitiv often drops the 'TR.' prefix in the returned DataFrame columns
price_col = "Price Close" if "Price Close" in raw_px.columns else "TR.PriceClose"

raw_px = raw_px.rename(columns={
    "Instrument": "ric",
    date_col: "date",
    price_col: "px" 
})

raw_px["date"] = pd.to_datetime(raw_px["date"])
raw_px["px"] = pd.to_numeric(raw_px["px"], errors="coerce")

px = (raw_px
      .dropna(subset=["date", "ric", "px"])
      .pivot_table(index="date", columns="ric", values="px")
      .sort_index()
      .ffill()
     )

px.shape

/opt/anaconda3/envs/final_project/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


ric         1COVG.F  1SXP.DE  1U1.DE  2GBG.DE  2HR.DE  8TRA.DE  A1OS.DE  \
date                                                                      
2026-02-09     61.0    14.16   26.35     37.2    4.32    35.98     34.1   
2026-02-10     61.0     14.7   25.45     37.4    4.36    36.18     35.0   
2026-02-11     61.0    15.64    25.5    37.45    4.49     36.8     37.7   
2026-02-12     61.0     14.7   24.85    36.25    4.44     36.3     37.5   
2026-02-13    60.76    14.98   24.45     36.8    4.41    36.46     37.3   

ric         AAGG.DE  AAHG.F  AALB.AS  ...  WLSNc.AS  WSU.DE  WUWGn.DE  \
date                                  ...                               
2026-02-09     15.2   0.012     34.8  ...     66.82    49.2     15.98   
2026-02-10    15.36  0.0115     35.6  ...     65.72    49.6     15.96   
2026-02-11    14.52  0.0145     35.2  ...     62.34    49.8     15.92   
2026-02-12    14.32   0.014    34.56  ...     60.48    49.6     15.78   
2026-02-13    14.32  0.0145    34.92

In [ ]:
rets = px.pct_change().dropna(how="all")

# chỉ giữ các mã có ESG + có returns
common = rets.columns.intersection(meta.index)
rets = rets[common]
meta2 = meta.loc[common].copy()

print("n_assets:", len(common))
rets.tail()


n_assets: 626


ric,1COVG.F,1SXP.DE,1U1.DE,2GBG.DE,2HR.DE,8TRA.DE,A1OS.DE,AAGG.DE,AAHG.F,AALB.AS,...,WLSNc.AS,WSU.DE,WUWGn.DE,XFAB.PA,XIOR.BR,YSNG.DE,ZALG.DE,ZILGn.DE,ZO1G.H,ZUMV.VI
date,,,,,,,,,,,,,,,,,,,,,
2026-02-09,0.004611,-0.002817,-0.014953,0.039106,-0.004608,-0.005528,-0.020115,0.049724,-0.714286,0.009281,...,-0.012999,-0.004049,0.017834,0.012587,-0.012238,-0.007335,0.005636,-0.007075,-0.058333,0.037448
2026-02-10,0.0,0.038136,-0.034156,0.005376,0.009259,0.005559,0.026393,0.010526,-0.041667,0.022989,...,-0.016462,0.00813,-0.001252,0.050645,0.00531,-0.004926,0.04624,-0.002375,-0.00885,0.069519
2026-02-11,0.0,0.063946,0.001965,0.001337,0.029817,0.017137,0.077143,-0.054688,0.26087,-0.011236,...,-0.05143,0.004032,-0.002506,0.011832,0.021127,-0.048515,-0.066964,0.017857,0.0,0.025
2026-02-12,0.0,-0.060102,-0.02549,-0.032043,-0.011136,-0.013587,-0.005305,-0.013774,-0.034483,-0.018182,...,-0.029836,-0.004016,-0.008794,-0.015591,-0.015517,-0.03642,-0.010526,-0.015205,0.0,0.02439
2026-02-13,-0.003934,0.019048,-0.016097,0.015172,-0.006757,0.004408,-0.005333,0.0,0.035714,0.010417,...,0.061177,0.012097,-0.016477,-0.011879,0.015762,0.017279,-0.008221,0.001188,0.0,-0.02381


In [ ]:
resA = run_backtest(rets, meta2, CFG, strategy_name="A")
print("A trades:", resA.trade_count, "final NAV:", float(resA.nav.iloc[-1]))

# Strategy B cần scipy; nếu bạn chưa cài scipy thì chạy A trước
resB = run_backtest(rets, meta2, CFG, strategy_name="B")
print("B trades:", resB.trade_count, "final NAV:", float(resB.nav.iloc[-1]))


A trades: 3969 final NAV: 1.1021973181047042
B trades: 1770 final NAV: 0.9899306006833509


In [ ]:
def max_drawdown(nav: pd.Series) -> float:
    peak = nav.cummax()
    dd = nav / peak - 1.0
    return float(dd.min())

print("A maxDD:", max_drawdown(resA.nav))
print("B maxDD:", max_drawdown(resB.nav))


A maxDD: -0.05822750940117394
B maxDD: -0.0753719742140444
